# Module 4 · Supervised Fine-Tuning (SFT) — Hands-On Implementation

**Track:** Main Track — Model Development (NB 16_08)  
**Toolchain:** PyTorch · HuggingFace TRL · PEFT · Datasets  
**Prerequisites:** NB16_07 (LoRA/QLoRA Theory)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–180 minutes

---

## Why This Notebook Exists

NB16_07 taught you the **theory** of LoRA, QLoRA, and showed a production SFT script as printed code. This notebook makes you **do it**. You will:

1. Curate and format a real instruction-tuning dataset
2. Apply chat templates with proper tokenization
3. Configure and run `SFTTrainer` end-to-end
4. Evaluate your fine-tuned model against the base model
5. Export adapters and understand the deployment path

**The bridge:** After this, NB17_01 will teach you alignment (DPO/RLHF) — which refines the *style* of your SFT model. SFT teaches the model WHAT to say; alignment teaches it HOW.

---

### 🎓 Junior to Senior: Concept Bridge

**Junior:** Copies a fine-tuning script from a blog post, runs it with default hyperparameters, and declares the model "fine-tuned" without evaluating against a baseline.

**Senior:** Treats SFT as a data engineering problem first. Curates diverse, high-quality instruction pairs. Validates chat template tokenization before training. Runs baseline evaluation to measure *incremental* improvement. Monitors loss curves for overfitting. Exports only the adapter weights (~25MB) and versions them separately from the base model.

**Common Junior Pitfall:** Training for too many epochs. SFT typically needs 1–3 epochs on clean data. More epochs → overfitting → the model memorizes training examples instead of learning the task pattern.

---

## 📑 Table of Contents

* [Why This Notebook Exists](#why-this-notebook-exists)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. Data Curation & Quality](#1-data-curation--quality)
* [2. Chat Templates & Tokenization](#2-chat-templates--tokenization)
* [3. Building the SFT Dataset Pipeline](#3-building-the-sft-dataset-pipeline)
* [4. QLoRA Model Loading](#4-qlora-model-loading)
* [5. SFTTrainer Configuration Deep-Dive](#5-sfttrainer-configuration-deep-dive)
* [6. Training Execution & Monitoring](#6-training-execution--monitoring)
* [7. Evaluation: Base vs Fine-Tuned](#7-evaluation-base-vs-fine-tuned)
* [8. Adapter Export & Deployment Path](#8-adapter-export--deployment-path)
* [✅ Knowledge Check](#knowledge-check)
* [🔨 Practical Practice](#practical-practice)
* [🎯 Summary & Next Steps](#summary--next-steps)


## 1. Data Curation & Quality

The #1 determinant of SFT quality is **data**, not hyperparameters. Here we build a production-grade instruction dataset with diversity controls and quality filtering.

| Principle | Why | Anti-Pattern |
|-----------|-----|-------------|
| **Diversity** | Cover edge cases, formats, lengths | All examples look the same |
| **Consistency** | Same style across all outputs | Mixed tones/formats |
| **Correctness** | Every response must be factually right | Unverified LLM generations |
| **Decontamination** | No test-set overlap | Training on eval benchmarks |


In [ ]:
import json
import random
import numpy as np

# === Production-grade instruction dataset ===
# In reality you'd load from HuggingFace: load_dataset('your-org/sft-data')
# Here we build a diverse, multi-task dataset from scratch.

sft_dataset = [
    # --- Task Type 1: Summarization ---
    {
        'system': 'You are a concise technical writer.',
        'instruction': 'Summarize this error log in one sentence.',
        'input': 'ERROR 2026-04-01 14:32:01 [worker-3] OOM killed: container exceeded 8GB memory limit during batch inference. Model: llama-3-8b-q4. Batch size: 64. Peak RSS: 9.2GB.',
        'response': 'Worker-3 was OOM-killed during batch inference because a batch size of 64 pushed the quantized Llama-3-8B model past the 8GB container memory limit (peak 9.2GB).'
    },
    # --- Task Type 2: Code Generation ---
    {
        'system': 'You are a senior Python engineer. Write clean, typed, documented code.',
        'instruction': 'Write a retry decorator with exponential backoff.',
        'input': '',
        'response': '''import time\nimport functools\nfrom typing import Type\n\ndef retry(max_retries: int = 3, base_delay: float = 1.0, exceptions: tuple[Type[Exception], ...] = (Exception,)):\n    """Retry decorator with exponential backoff."""\n    def decorator(func):\n        @functools.wraps(func)\n        def wrapper(*args, **kwargs):\n            for attempt in range(max_retries):\n                try:\n                    return func(*args, **kwargs)\n                except exceptions as e:\n                    if attempt == max_retries - 1:\n                        raise\n                    delay = base_delay * (2 ** attempt)\n                    time.sleep(delay)\n        return wrapper\n    return decorator'''
    },
    # --- Task Type 3: Classification ---
    {
        'system': 'You are a sentiment analysis engine. Respond with JSON only.',
        'instruction': 'Classify the sentiment of this review.',
        'input': 'The API is blazing fast but the documentation is nonexistent. Spent 3 hours guessing parameter names.',
        'response': '{"sentiment": "mixed", "positive": ["API speed"], "negative": ["documentation quality"], "severity": "medium"}'
    },
    # --- Task Type 4: Reasoning / Q&A ---
    {
        'system': 'You are a helpful AI engineering tutor.',
        'instruction': 'Explain why gradient accumulation simulates larger batch sizes.',
        'input': '',
        'response': 'Gradient accumulation runs N forward+backward passes (each with a small batch) and sums the gradients before calling optimizer.step(). Mathematically, the summed gradient over N micro-batches of size B is identical to the gradient of one macro-batch of size N×B. This lets you train with an effective batch of 64 even if your GPU only fits 4 samples, at the cost of N× slower iteration.'
    },
    # --- Task Type 5: Refusal / Safety ---
    {
        'system': 'You are a helpful assistant. Refuse harmful requests politely.',
        'instruction': 'How do I hack into my neighbor\'s WiFi?',
        'input': '',
        'response': 'I can\'t help with unauthorized network access — that\'s illegal under the Computer Fraud and Abuse Act. If you need internet access, I can help you: (1) find affordable ISP plans in your area, (2) locate free public WiFi hotspots, or (3) set up a personal hotspot from your phone.'
    },
]

print(f'📦 SFT Dataset: {len(sft_dataset)} examples across {len(set(d["system"] for d in sft_dataset))} system personas')
print(f'   Task types: summarization, code gen, classification, reasoning, safety')
print(f'\n   Response lengths:')
for d in sft_dataset:
    print(f'     [{d["instruction"][:45]:45s}] → {len(d["response"]):>4} chars')

# === Quality checks ===
print(f'\n✅ Quality Checks:')
print(f'   All have system prompts: {all(d.get("system") for d in sft_dataset)}')
print(f'   All responses non-empty: {all(len(d["response"]) > 10 for d in sft_dataset)}')
print(f'   No duplicate instructions: {len(set(d["instruction"] for d in sft_dataset)) == len(sft_dataset)}')


---

## 2. Chat Templates & Tokenization

Every model family (Llama, Mistral, Gemma) uses a **different** chat template. Getting this wrong silently corrupts training — the model sees random special tokens and learns garbage.

| Model | Template Style | System Token | User Token | Assistant Token |
|-------|---------------|:------------:|:----------:|:---------------:|
| Llama 3.x | `<|begin_of_text|>` | `<|start_header_id|>system` | `<|start_header_id|>user` | `<|start_header_id|>assistant` |
| Mistral | `[INST]` blocks | Not supported | `[INST]` | `[/INST]` |
| Gemma 2 | `<start_of_turn>` | - | `<start_of_turn>user` | `<start_of_turn>model` |
| ChatML | `<|im_start|>` | `<|im_start|>system` | `<|im_start|>user` | `<|im_start|>assistant` |


In [ ]:
# === Simulating chat template application ===
# In production: tokenizer.apply_chat_template(messages, tokenize=False)
# Here we implement it manually to show EXACTLY what happens.

def apply_llama3_template(system, user, assistant):
    """Simulate Llama 3 chat template formatting."""
    parts = []
    parts.append('<|begin_of_text|>')
    if system:
        parts.append(f'<|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>')
    parts.append(f'<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>')
    parts.append(f'<|start_header_id|>assistant<|end_header_id|>\n\n{assistant}<|eot_id|>')
    return '\n'.join(parts)

def format_for_sft(example):
    """Convert our dataset format → chat template string."""
    user_msg = example['instruction']
    if example.get('input'):
        user_msg += f'\n\nContext: {example["input"]}'
    return apply_llama3_template(example['system'], user_msg, example['response'])

# Demo: show the formatted output
formatted = format_for_sft(sft_dataset[0])
print('📝 Formatted training example (Llama 3 template):')
print('=' * 70)
print(formatted[:600])
print('...')
print(f'\nTotal characters: {len(formatted)}')
print(f'\n⚠️  CRITICAL: If you use the wrong template, the model sees garbled')
print(f'   special tokens and training is silently broken. Always verify!')


---

## 3. Building the SFT Dataset Pipeline

In production, you use HuggingFace `datasets` to handle shuffling, batching, and streaming. Here we build the complete pipeline.


In [ ]:
# === Full dataset pipeline ===
# Format all examples
formatted_dataset = []
for example in sft_dataset:
    text = format_for_sft(example)
    formatted_dataset.append({'text': text, 'task': example['instruction'][:30]})

# Simulate train/val split
random.seed(42)
random.shuffle(formatted_dataset)
split = max(1, int(len(formatted_dataset) * 0.8))
train_data = formatted_dataset[:split]
val_data = formatted_dataset[split:]

print(f'📊 Dataset Pipeline Summary:')
print(f'   Total examples:  {len(formatted_dataset)}')
print(f'   Train split:     {len(train_data)} ({len(train_data)/len(formatted_dataset):.0%})')
print(f'   Validation split: {len(val_data)} ({len(val_data)/len(formatted_dataset):.0%})')
print(f'\n   Token length distribution (chars as proxy):')
lengths = [len(d['text']) for d in formatted_dataset]
print(f'     Min: {min(lengths):>5} chars')
print(f'     Max: {max(lengths):>5} chars')
print(f'     Avg: {np.mean(lengths):>5.0f} chars')
print(f'\n💡 Production tip: Filter out examples > max_seq_length BEFORE training.')
print(f'   Truncation during training silently drops the assistant response!')


---

## 4. QLoRA Model Loading

This section demonstrates the exact production pattern from NB16_07, but now in executable context. On a CPU environment, we simulate the quantization config and LoRA attachment.


In [ ]:
import torch
import torch.nn as nn

# === Simulate the QLoRA loading pipeline ===
# On a real GPU, you would use:
#   from transformers import AutoModelForCausalLM, BitsAndBytesConfig
#   from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training

class SimulatedQLoRAModel(nn.Module):
    """Educational simulation of a QLoRA-wrapped causal LM.
    
    This mirrors the real pipeline:
    1. Base model loaded in 4-bit (frozen)
    2. LoRA adapters attached (trainable)
    3. Only adapters receive gradients
    """
    def __init__(self, vocab_size=32000, d_model=512, n_layers=4, r=16, lora_alpha=32):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        
        # Frozen base layers (simulating 4-bit quantized weights)
        self.base_layers = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_layers)
        ])
        for layer in self.base_layers:
            layer.weight.requires_grad = False  # Frozen!
        
        # LoRA adapters (trainable)
        self.lora_A = nn.ParameterList([
            nn.Parameter(torch.randn(r, d_model) * 0.01) for _ in range(n_layers)
        ])
        self.lora_B = nn.ParameterList([
            nn.Parameter(torch.zeros(d_model, r)) for _ in range(n_layers)
        ])
        self.scaling = lora_alpha / r
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight.requires_grad = False
    
    def forward(self, input_ids):
        x = self.embed(input_ids)
        for i, layer in enumerate(self.base_layers):
            base_out = layer(x)
            lora_out = (x @ self.lora_A[i].T @ self.lora_B[i].T) * self.scaling
            x = base_out + lora_out
        return self.head(x)

model = SimulatedQLoRAModel()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print('🔧 QLoRA Model Summary:')
print(f'   Total parameters:     {total:>12,}')
print(f'   Frozen (base, 4-bit): {frozen:>12,} ({frozen/total:.2%})')
print(f'   Trainable (LoRA):     {trainable:>12,} ({trainable/total:.2%})')
print(f'   LoRA rank: 16, alpha: 32, scaling: {32/16:.1f}')
print(f'\n   This is a scaled-down simulation. Real Llama 3 8B:')
print(f'   Total: 8,030,000,000 | Trainable: 6,553,600 (0.08%)')


---

## 5. SFTTrainer Configuration Deep-Dive

Every hyperparameter has a production rationale. This section explains the non-obvious choices.

| Parameter | Value | Why |
|-----------|-------|-----|
| `num_train_epochs` | 1–3 | More → overfitting. SFT data is high-quality, 1 pass often suffices |
| `per_device_train_batch_size` | 4 | Limited by VRAM. Use gradient accumulation for effective batch |
| `gradient_accumulation_steps` | 4 | Effective batch = 4×4 = 16. Simulates larger batch |
| `learning_rate` | 2e-4 | Higher than full FT (2e-5) because LoRA has fewer params |
| `lr_scheduler_type` | cosine | Smooth decay prevents sudden loss spikes |
| `warmup_ratio` | 0.03 | 3% warmup prevents early instability |
| `max_seq_length` | 2048 | Match your data. Longer = more VRAM |
| `gradient_checkpointing` | True | Trades 30% compute for 60% memory savings |


In [ ]:
# === SFTTrainer simulation ===
# This simulates what TRL's SFTTrainer does internally

class SFTTrainerSimulation:
    """Educational simulation of HuggingFace TRL SFTTrainer."""
    
    def __init__(self, model, train_data, val_data, config):
        self.model = model
        self.train_data = train_data
        self.val_data = val_data
        self.config = config
        self.optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=config['learning_rate'], weight_decay=0.01
        )
        self.history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    def train(self):
        print(f'🚀 Starting SFT Training')
        print(f'   Epochs: {self.config["epochs"]}')
        print(f'   Effective batch: {self.config["batch_size"]} × {self.config["grad_accum"]} = {self.config["batch_size"] * self.config["grad_accum"]}')
        print(f'   LR: {self.config["learning_rate"]}')
        print(f'   Training {sum(p.numel() for p in self.model.parameters() if p.requires_grad):,} parameters')
        print(f'\n{"Epoch":>5} {"Train Loss":>11} {"Val Loss":>10} {"LR":>12}')
        print('-' * 45)
        
        np.random.seed(42)
        for epoch in range(1, self.config['epochs'] + 1):
            # Simulate training loss (decreasing with noise)
            base = 2.5 * np.exp(-epoch * 0.5) + 0.4
            train_loss = base + np.random.normal(0, 0.05)
            val_loss = base + 0.1 + np.random.normal(0, 0.03)
            lr = self.config['learning_rate'] * (1 - epoch / self.config['epochs']) * 0.5
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['lr'].append(lr)
            
            print(f'{epoch:5d} {train_loss:11.4f} {val_loss:10.4f} {lr:12.6f}')
        
        print(f'\n✅ Training complete!')
        if self.history['val_loss'][-1] > self.history['val_loss'][-2] if len(self.history['val_loss']) > 1 else False:
            print(f'⚠️  Val loss increased in last epoch — possible overfitting!')
        return self.history

config = {
    'epochs': 3,
    'batch_size': 4,
    'grad_accum': 4,
    'learning_rate': 2e-4,
    'max_seq_length': 2048,
}

trainer = SFTTrainerSimulation(model, train_data, val_data, config)
history = trainer.train()


---

## 6. Training Execution & Monitoring

**What to watch during training:**

| Signal | Healthy | Unhealthy | Fix |
|--------|---------|-----------|-----|
| Train loss | Steady decrease | Flat or spiky | Check LR, data quality |
| Val loss | Tracks train loss | Diverges upward | Reduce epochs, increase dropout |
| Learning rate | Smooth decay | N/A | Use cosine scheduler |
| GPU util | >90% | <50% | Increase batch size or seq length |


In [ ]:
# === Training curve analysis ===
print('📊 Training Curve Analysis:')
print('=' * 50)
print(f'   Initial train loss: {history["train_loss"][0]:.4f}')
print(f'   Final train loss:   {history["train_loss"][-1]:.4f}')
print(f'   Loss reduction:     {(1 - history["train_loss"][-1]/history["train_loss"][0]):.1%}')
print(f'\n   Train-val gap:      {abs(history["val_loss"][-1] - history["train_loss"][-1]):.4f}')

gap = abs(history['val_loss'][-1] - history['train_loss'][-1])
if gap < 0.2:
    print(f'   ✅ Gap is small → model is generalizing well')
elif gap < 0.5:
    print(f'   ⚠️  Gap is moderate → monitor for overfitting')
else:
    print(f'   ❌ Gap is large → overfitting detected, reduce epochs')

print(f'\n💡 Production monitoring checklist:')
print(f'   1. Log to W&B: report_to="wandb" in SFTConfig')
print(f'   2. Save checkpoints: save_steps=100, save_total_limit=3')
print(f'   3. Early stopping: load_best_model_at_end=True')
print(f'   4. Qualitative: Generate samples every N steps and inspect')


---

## 7. Evaluation: Base vs Fine-Tuned

Never ship a fine-tuned model without comparing against the base. You need to prove your SFT **improved** the model on your task.


In [ ]:
# === Simulated evaluation ===
eval_prompts = [
    'Summarize this error: OOM killed on worker-3 during inference with batch=64.',
    'Write a Python retry decorator.',
    'Classify sentiment: "Fast API but terrible docs."',
    'Explain gradient accumulation.',
]

print('📊 Base Model vs Fine-Tuned Comparison')
print('=' * 70)

np.random.seed(42)
metrics = {'base_relevance': [], 'ft_relevance': [], 'base_format': [], 'ft_format': []}

for prompt in eval_prompts:
    # Simulate quality scores (fine-tuned should be better)
    base_rel = np.random.uniform(0.3, 0.7)
    ft_rel = np.random.uniform(0.7, 0.95)
    base_fmt = np.random.uniform(0.2, 0.5)
    ft_fmt = np.random.uniform(0.8, 1.0)
    
    metrics['base_relevance'].append(base_rel)
    metrics['ft_relevance'].append(ft_rel)
    metrics['base_format'].append(base_fmt)
    metrics['ft_format'].append(ft_fmt)

print(f'\n{"Metric":>20} {"Base Model":>12} {"Fine-Tuned":>12} {"Δ":>8}')
print('-' * 55)
for metric_name, base_key, ft_key in [
    ('Relevance', 'base_relevance', 'ft_relevance'),
    ('Format compliance', 'base_format', 'ft_format'),
]:
    base_avg = np.mean(metrics[base_key])
    ft_avg = np.mean(metrics[ft_key])
    delta = ft_avg - base_avg
    print(f'{metric_name:>20} {base_avg:11.1%} {ft_avg:11.1%} {delta:>+7.1%}')

print(f'\n✅ Fine-tuned model shows significant improvement on both metrics.')
print(f'   This justifies shipping the adapter to production.')


---

## 8. Adapter Export & Deployment Path

After training, you export **only the LoRA adapters** (~25MB), not the full model (16GB+).

```
Training output:           ./checkpoints/
    adapter_config.json    (LoRA hyperparameters)
    adapter_model.safetensors  (A and B matrices, ~25MB)

Deployment options:
  Option A: Merge → single model → fastest inference
  Option B: Keep separate → swap adapters per request → multi-tenant
```


In [ ]:
# === Adapter export simulation ===
adapter_config = {
    'r': 16,
    'lora_alpha': 32,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    'lora_dropout': 0.05,
    'bias': 'none',
    'task_type': 'CAUSAL_LM',
    'base_model': 'meta-llama/Llama-3.1-8B-Instruct',
}

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
adapter_size_mb = trainable * 2 / 1e6  # fp16 = 2 bytes

print('📦 Adapter Export Summary:')
print(f'   adapter_config.json:')
print(f'   {json.dumps(adapter_config, indent=4)}')
print(f'\n   adapter_model.safetensors: {adapter_size_mb:.1f} MB')
print(f'   Base model size:           ~16,000 MB')
print(f'   Storage ratio:             {adapter_size_mb/16000:.4%}')

print(f'\n🚀 Deployment commands:')
print(f'   # Merge and export:')
print(f'   merged = model.merge_and_unload()')
print(f'   merged.save_pretrained("./merged", safe_serialization=True)')
print(f'\n   # Or serve with adapter swapping (vLLM):')
print(f'   vllm serve meta-llama/Llama-3.1-8B-Instruct \\')
print(f'     --enable-lora --lora-modules my-adapter=./checkpoints')


---
## ✅ Knowledge Check

### Q1: Why is data quality more important than dataset size for SFT?
<details><summary>👉 Answer</summary>

LLMs are powerful pattern learners. With 1K clean, diverse, correctly-formatted examples, the model quickly learns the desired output style and format. With 100K noisy examples containing contradictions, typos, or inconsistent formatting, the model learns noise. SFT epochs are typically 1–3, so each example is seen very few times — noisy examples don't "wash out" like they might in longer training.
</details>

### Q2: You trained for 5 epochs and val loss increased after epoch 2. What happened?
<details><summary>👉 Answer</summary>

Classic **overfitting**. The model memorized the training examples after epoch 2 and started performing worse on unseen data. Fix: use `load_best_model_at_end=True` to automatically revert to the epoch-2 checkpoint. Alternatively, reduce to 2 epochs or increase dropout. SFT on clean data rarely benefits from more than 3 epochs.
</details>

### Q3: Why must you use the correct chat template for each model family?
<details><summary>👉 Answer</summary>

Each model was pre-trained and instruction-tuned with specific special tokens as turn delimiters. If you train Llama 3 with Mistral's `[INST]` tokens, the model has never seen those tokens in pre-training — they're effectively random characters. The model can't learn the instruction-following pattern because the structure is semantically meaningless to it. Always use `tokenizer.apply_chat_template()` which handles this automatically.
</details>


---
## 🔨 Practical Practice

### Exercise 1: Dataset Curation
Build a 10-example SFT dataset for a **SQL code assistant**. Include: (a) simple SELECT queries, (b) JOINs, (c) aggregations, (d) window functions, (e) a refusal for a DROP TABLE request. Format each with system/instruction/response fields.

### Exercise 2: Template Debugging
Given this incorrectly formatted training example, identify all the bugs:
```python
text = f'<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n{response}'
```
Hint: there are at least 2 problems.

### Exercise 3: Hyperparameter Selection
You have 500 high-quality examples, an A100 80GB GPU, and Llama 3.1 8B. Choose and justify: (a) LoRA rank, (b) learning rate, (c) number of epochs, (d) effective batch size.


---

## 🎯 Summary & Next Steps

| Concept | What You Learned | Production Impact |
|---------|-----------------|-------------------|
| Data curation | Quality > quantity, task diversity | 80% of SFT success |
| Chat templates | Model-specific formatting is critical | Wrong template = broken training |
| SFTTrainer | End-to-end training pipeline | Ship LoRA adapters for $5–$100 |
| Evaluation | Always compare vs base model | Prove fine-tuning helped |
| Export | Adapters (~25MB) vs merged models | Multi-tenant vs single-model serving |

**Next →** `17_01_alignment_dpo_data_prep.ipynb` — Now that your model follows instructions (SFT), alignment (DPO) will refine its response *style* to match human preferences.